In [1]:
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point
import matplotlib.pyplot as plt
from shapely.geometry import Point
import pandas as pd
from plotnine import *
from geopandas import GeoDataFrame
import contextily as cx

%matplotlib inline

In [2]:
# converting to points 
County = "Dallas"
county = "dallas"

df = pd.read_csv(f"../{County}_Data/cleaned_{county}_snap_data.csv")

# Create point geometries
geometry = gpd.points_from_xy(df["Longitude"], df["Latitude"])

geo_df = gpd.GeoDataFrame(
    df[["Store_Name", "Store_Street_Address", "City", "Zip_Code", "Store_Type", "Latitude", "Longitude"]], 
    geometry=geometry,
    crs="EPSG:4326"
)

geo_df.head()

,Store_Name,Store_Street_Address,City,Zip_Code,Store_Type,Latitude,Longitude,geometry
0,HEB Food Store 817,2351 W Interstate Highway 635,Irving,75063,Supermarket,32.921303,-96.983322,POINT (-96.98332 32.9213)
1,Homefoods I I I Llc,2301 W Rochelle Rd,Irving,75062,Grocery Store,32.841888,-96.975380,POINT (-96.97538 32.84189)
2,Fair Park Farmers Market,3535 Grand Ave,Dallas,75210,Farmers and Markets,32.778282,-96.763084,POINT (-96.76308 32.77828)
3,Joe V's Smart Shop 805,7700 Samuell Blvd,Dallas,75227,Supermarket,32.792328,-96.685730,POINT (-96.68573 32.79233)
4,Abyssinia Market Irving,4010 N Belt Ln Rd,Irving,75038,Grocery Store,32.867268,-96.981628,POINT (-96.98163 32.86727)


In [3]:
farms_df = pd.read_csv(f"../final_urbanfarm_data/final_urban_farms_{county}.csv")

farms_gdf = gpd.GeoDataFrame(
    farms_df,
    geometry=gpd.points_from_xy(farms_df["lon"], farms_df["lat"]),
    crs="EPSG:4326"  # standard lat/lon
)

In [4]:
counties = gpd.read_file("../Census_Tract/tl_2025_48_tract.shp")

In [5]:
counties = counties[counties["COUNTYFP"] == "113"]
roads = GeoDataFrame.from_file("../OpenStreetMap/gis_osm_roads_free_1.shp", encoding="utf-8")

In [6]:
# reproject everything to Web Mercator 
counties_wm = counties.to_crs(epsg=3857)
geo_df_wm = geo_df.to_crs(epsg=3857)
roads_wm = roads.clip(counties.total_bounds).to_crs(epsg=3857)  
farms_gdf_wm = farms_gdf.to_crs(epsg=3857)

In [ ]:
# buffer (in meters, since Web Mercator units are meters at this scale)
geo_df_buffer_wm = geo_df_wm.copy()
geo_df_buffer_wm["geometry"] = geo_df_wm.geometry.buffer(1609)  # ~1 mile

# plot 
colors = {
    "Grocery Store": "green",
    "Supermarket": "blue",
    "Super Store": "purple",
    "Farmers and Markets": "orange",
    "Other": "red",
}

ig, ax = plt.subplots(figsize=(14, 12))

roads_wm.plot(ax=ax, color="gray", linewidth=0.3, alpha=0.4)
geo_df_buffer_wm.plot(ax=ax, color="steelblue", alpha=0.15, edgecolor="blue", linewidth=0.3)

for store_type, color in colors.items():
    subset = geo_df_wm[geo_df_wm["Store_Type"] == store_type]
    subset.plot(ax=ax, marker="o", color=color, markersize=8, alpha=0.7, label=store_type)

farms_gdf_wm.plot(ax=ax, color="#D2691E", marker="^", markersize=100,
                   edgecolor="#D2691E", linewidth=1.0, zorder=4)
    
ax.set_xlim(counties_wm.total_bounds[[0, 2]])
ax.set_ylim(counties_wm.total_bounds[[1, 3]])

cx.add_basemap(ax, source=cx.providers.CartoDB.Positron, zorder=0)  # now the extent is correct

ax.legend(title="Store Type", fontsize=10, title_fontsize=11, loc="lower right")

ax.axis("off")
ax.set_title(f"SNAP Grocery Stores and Urban Farms in {County} County", fontsize=16, fontweight="bold", pad=5)
plt.tight_layout()

plt.savefig(f"{county}_snap_farms_map.png", dpi=300, bbox_inches="tight")
plt.show()

In [7]:
print(farms_gdf.crs)
print(farms_gdf.total_bounds)

EPSG:4326
[-98.55616822  29.2322501  -98.1879184   29.57      ]
